# 0.3 · The LGD prior

*0. General · notebook 0.3 of the story.* ← [0.2 · the PD prior](<0.2_prior_visualisation_pd.ipynb>) · [1.1 · PD training](<../1. Experiment 1/1.1_pd_training.ipynb>) →

**What does our credit prior generate for LGD, how does it differ from TabICL's own, and does it look
like the real books of 0.1?** The target is a loss share bounded in [0, 1], with mass at both ends. Experiment 1 then trains on both priors and lets
the benchmark decide; this notebook checks, before that compute is spent, that the prior is worth
training on — that it looks like real data, that each credit mechanism produces the structure it
claims, and that the task it sets is the right difficulty.

Every figure generates live from `config/Exp1_LGD.yaml` through `TaskGenerator` — the exact code path
training uses — or reads pre-generated pools when they are on disk. **Original** is the prior at
`credit_fraction = 0`: TabICL's `graph_scm`, unchanged, and exactly Experiment 1's control. **Credit** is
ours, at `credit_fraction = 1`.

**How to read it.** Part A compares the two priors against the real data. Part B takes our prior apart,
one credit mechanism at a time, to show where each difference comes from. Part C checks that the task is
learnable and the tables sane. Figures that answered nothing have been removed rather than kept for
volume.

## Grounding in the literature

LGD is regression on a bounded [0, 1] target with mass at the boundaries. TabICLv2 standard-scales the
regression target — affine, so it destroys the [0, 1] support but not bimodality — and produces boundary
atoms only **by accident**: the 4σ outlier clamp sets every outlier to one value, manufacturing ties at
the extremes (`repositories/TabICL.txt` `outlier_removing`). Our prior builds the atoms **by
construction**, at 0 (full recovery) and 1 (total loss). Representability is not the obstacle: the head
is a 999-quantile pinball predictor (`papers/2026/02_Qu_TabICLv2` §I), which represents a point mass
exactly, and the prior already carries a Kumaraswamy [0, 1] warp (`repositories/NanoTabICL.txt`
`rand_kumaraswamy_act`). O'Prior supports but never evaluates bounded regression
(`papers/2026/05_Bouadi_ShapingThePrior`), so this is genuinely open ground; the primary metrics are
distributional, because CRPS scores the whole predictive (`SYNTHESIS.md`). Pin `e5ce016`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder
# (notebooks/1. Experiment 1/), from notebooks/, or from the root — then work FROM the root,
# so relative paths (config/...) resolve exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import exp1_plots as e1, figures, pool_plots as pp, mechanism_plots, prior_plots, style, summaries, literature

style.apply()   # ONE shared style: identical colours in every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK   = "lgd"
CONFIG = "config/Exp1_LGD.yaml"   # Exp1 is the prior sweep, so it is the one to visualise
N      = 500     # datasets to draw per prior - enough to show the structure
FOCUS  = None    # variant for the detail plots; None = the first non-original one

# Clears THIS notebook's figure folder - and no other - BEFORE anything is drawn, then saves
# every figure as a PDF sized for A4. Identical in Jupyter and under the batch runner.
FIGS = figures.FigureSaver("0.3_prior_visualisation_lgd")

## The colour key

One vocabulary for every figure in every notebook. Blue is our credit prior, grey TabICL's unmodified
prior, orange real data, magenta a real-data marker drawn over coloured points, teal a value from
`tfm-library` and amber one from outside it; the two shades of blue are the mild and aggressive
settings of our prior.

In [ ]:
FIGS.save(
    style.show_palette(),
    "palette",
    caption=(
"Colour key. Grey denotes the unmodified TabICL prior, blue the credit-targeted "
"prior, orange values measured from the real datasets, and red out-of-range or "
"flagged values."
    ),
);


## Loading the priors and the real data

Pre-generated pools if any are on disk, otherwise both arms generated live — the printed `source` says
which. The real LGD datasets are loaded alongside: without them every figure would describe the prior in
isolation.

In [ ]:
variants = pp.discover_pools(TASK)
print("pools found:", ", ".join(variants) if variants else "none - will generate live")
loaded, SOURCE = pp.load_variants_or_generate(TASK, n=N, seed=0, config=CONFIG)
print("source =", SOURCE)
print({k: len(v) for k, v in loaded.items()})

FOCUS = FOCUS or next((k for k in loaded if not k.startswith("original")), list(loaded)[0])
print("FOCUS (detail plots) =", FOCUS)

# The real datasets, for every comparison below. Without them the figures describe the prior in
# isolation, which is the thing that made the old notebook uninformative.
REAL = summaries.load_real_datasets(TASK)
print(f"real {TASK.upper()} datasets loaded: {len(REAL)}")
REFERENCE = pp.real_reference(TASK)


## A · Does our prior look like real LGD data?

The two priors side by side, against the real books — where the boundary mass sits, how close the whole target distribution comes, and the shapes themselves.

### A1 · The two priors in numbers

Table shape, target statistics and how each prior sits against the real datasets, before any figure.
A difference in table shape between the priors would be a confound rather than a finding (C4 checks it).

In [ ]:
pp.variant_summary(loaded, TASK)


### A2 · Which boundary does the mass sit on?

Total boundary mass hides the asymmetry that matters. A book where most defaults recover in full is a
different book from one where most are a total loss, and both can share the same total. The real
datasets are the targets to land on.

**How to read it.** Magenta stars are the real datasets; coloured dots are synthetic ones. A prior lands well if its cloud sits where the stars are. Above the dotted diagonal means loss-heavy, below it recovery-heavy.

In [ ]:
FIGS.save(
        e1.plot_boundary_mass_sources(loaded, REAL),
        "boundary_mass_sources",
        caption=(
"Share of rows at exactly 0 against share at exactly 1, one point per synthetic "
"dataset and one panel per prior variant. Stars mark the real LGD datasets. The "
"dotted diagonal is equal mass at both ends."
        ),
    );


### A3 · Which prior looks most like real LGD data?

For each prior, how far its target distribution sits from each real LGD dataset's, as a total-variation
distance — lower is better. The spread of the dots matters as much as the mean: a prior that matches one
dataset and misses the rest is not a good prior. O'Prior's generator supports regression but its whole
evaluation is classification (`papers/2026/05_Bouadi_ShapingThePrior`), so there is no published
bounded-target realism to compare against — this is the gap the project works in.

**How to read it.** One row per prior, best at the top. Grey dots are the individual real datasets and the blue diamond their mean, so a wide spread means the prior matches some datasets and misses others — worse than a slightly larger mean with dots close together.

In [ ]:
FIGS.save(
    e1.plot_prior_realism_ranking(loaded, REAL, task=TASK),
    "prior_realism_ranking",
    caption=(
"Distance between each prior variant's pooled target distribution and each real LGD "
"dataset's, as total variation over 40 fixed bins. Diamonds give the mean across "
"datasets, dots one per dataset. Variants ordered by mean distance."
    ),
);


### A4 · Target shapes, one row per prior

Kept for LGD because the *shape* of a bounded target is genuinely informative — U, J or a flat interior,
and how the atoms sit against it. Paginated: ten panels across A4 would be 0.63 in each.

**How to read it.** Rows share a draw index, so panels in the same column come from the same random seed and are directly comparable across priors.

In [ ]:
for _page in range(1, pp.shape_pages(n_per=10) + 1):
        FIGS.save(
            pp.plot_target_shapes_by_variant(loaded, n_per=10, page=_page),
            f"target_shapes_by_variant_p{_page}",
            caption=(
"Histograms of the target for ten synthetic datasets per prior variant, one variant "
"per row, 25 bins per panel. Rows use the same draw index, so panels in the same "
"column are directly comparable."
                f" Page {_page} of {pp.shape_pages(n_per=10)}."
            ),
        )


## B · Where the difference comes from

Our prior taken apart: where its boundary atoms come from, how strongly it pushes them, and the three mechanisms it shares with PD — shift, informative missingness and the filter.

### B1 · Where the boundary atoms come from

**The central claim as a picture.** Our prior derives the loss from a credit story — collateral, workout,
or a mixture of portfolio segments — so the mass at 0 and at 1 is a *consequence* of that story rather
than a parameter. If the claim holds, `collateral` should own most of the atom at 0: an
over-collateralised loan recovers in full by construction. The original prior's atoms, by contrast, come
from the 4σ clamp (`repositories/TabICL.txt` `outlier_removing`) — ties at an arbitrary scale.

**How to read it.** Each panel is one loss story. The spikes at 0 and 1 are what we claim *emerges* from the economics: `collateral` should own most of the mass at 0, because an over-collateralised loan recovers in full by construction.

In [ ]:
FIGS.save(
        e1.plot_mechanism_decomposition(loaded[FOCUS]),
        "mechanism_decomposition",
        caption=(
"Distribution of the LGD target, split by the loss mechanism that generated each "
"synthetic dataset, 40 bins per panel on a fixed [0,1] support. Percentages give the "
"share of rows lying exactly at 0 and exactly at 1. Panel subtitles give the number "
"of datasets per mechanism."
        ),
    );


### B2 · Boundary-atom intensity

The one lever Experiment 1 sweeps for LGD intensity: the range the boundary mass is drawn from — mild
(0.02–0.30) against aggressive (0.15–0.60) — beside the original prior. Each panel pools every task's
target, so the spikes at 0 and 1 grow from left to right; the subtitle gives the mean boundary mass.
Compare it with the real books' boundary mass in 0.1 A3.

In [ ]:
FIGS.save(
    mechanism_plots.intensity_atoms(CONFIG, n=50),
    'adj_intensity_atoms',
    caption=(
        "Histogram of the pooled LGD target over ~50 synthetic tasks from each prior: the original TabICL prior (grey) and our prior at mild (light blue) and aggressive (dark blue) boundary intensity. Spikes at 0 and 1 are the boundary atoms; the panel subtitle gives the mean total boundary mass."
    ),
);

### B3 · Distribution shift — context versus query

Deployment is never i.i.d.: the losses scored today come from a later cohort or a different mix than the
ones the model conditions on. Each panel switches on one kind of shift — cohort (the whole book drifts),
covariate (a feature's range moves), prior probability (the loss level moves) — and compares context
with query. Purucker 2026 shows TFMs lose ground off the i.i.d. regime (`papers/2026/06_Purucker_BeyondIID`).

In [ ]:
FIGS.save(
    mechanism_plots.shift_kinds(CONFIG, n=40),
    'adj_shift_kinds',
    caption=(
        "LGD target in the context rows (grey) versus the query rows (orange), one panel per distribution-shift kind, ~40 tasks each with that shift forced on. Cohort and prior-probability shifts move the target; covariate shift moves the shown feature instead, leaving the target relationship intact."
    ),
);

### B4 · Informative missingness (MNAR)

A missing value is itself a signal. Under the MNAR coupling the missing rate rises with the loss; under
MCAR it is flat. TabICLv2 mean-imputes missing values away at inference (`repositories/TabICL.txt`
`TransformToNumerical`), and O'Prior models MNAR explicitly (`papers/2026/05_Bouadi_ShapingThePrior`
§2.2) — the mechanism reproduced here.

In [ ]:
FIGS.save(
    mechanism_plots.informative_missingness(TASK),
    'adj_informative_missingness',
    caption=(
        "Missing rate as a function of the LGD outcome for a controlled dataset, under missing-completely-at-random (grey, coupling 0) and target-coupled missingness (blue, coupling 2). Under coupling the rate rises with the outcome; under MCAR it is flat."
    ),
);

### B5 · The predictability filter

Which generated tasks are kept. A 25-tree ExtraTrees must beat the mean at a bootstrap p < 0.05 or the
task is rejected (`repositories/TabICL.txt` `should_filter`); TabICLv2 rejects ~25 % of regression tasks
in stage 1 and reports it aids convergence (`papers/2026/02_Qu_TabICLv2` §Data filtering, Fig. 10). It is
a significance test, not an R² floor (§E.14) — a weak-but-real signal at n ≈ 1024 usually passes, so it
removes no-signal tasks more than low-signal ones. That is why `banded` targets the low-signal band
directly, the shaded region, rather than relying on the filter.

In [ ]:
FIGS.save(
    mechanism_plots.filter_modes(CONFIG, n=60),
    'adj_filter_modes',
    caption=(
        "Distribution of ExtraTrees pseudo-R^2 over ~60 generated LGD tasks. The shaded band is what filter mode 'banded' keeps; 'off' keeps all, 'tabicl' keeps the predictable tail."
    ),
);

## C · Is it a learnable task, and are the tables sane?

Difficulty, what the model literally sees, the feature structure, and the table shapes.

### C1 · Is the synthetic task the right difficulty?

A prior whose tasks are trivially easy teaches the model that features determine the target exactly;
one whose tasks are noise teaches it to predict the mean. Real credit data is neither — **low signal but
not zero** — and the shaded band is that target, measured with the filter's own small-ExtraTrees family.

**How to read it.** The shaded band is where real credit data sits. A prior far above it is too easy and teaches the model that features determine the target almost exactly; far below is noise.

In [ ]:
REAL_SCORES = summaries.real_difficulty(TASK, REAL)
print("real-data difficulty:", {k: round(v, 3) for k, v in REAL_SCORES.items()})

FIGS.save(
    e1.plot_difficulty_calibration(loaded, REAL_SCORES, task=TASK),
    "difficulty_calibration",
    caption=(
"Predictability of each synthetic dataset under a small ExtraTrees on a 70/30 split, "
"one point per dataset and one column per prior variant, with the median marked. The "
"shaded band spans the same measurement on the real credit datasets."
    ),
);


### C2 · What does the model actually see?

Every figure above is a summary statistic. This is the thing itself: one synthetic table and one real
table, same layout, a few rows each.

**How to read it.** Shade is the value's rank within its own column, so compare *texture* — how much variation, how many repeats — not individual cells.

In [ ]:
_real_one = next(iter(REAL.values())) if REAL else None
FIGS.save(
    e1.plot_side_by_side_tables(loaded[FOCUS][0], _real_one, task=TASK),
    "side_by_side_tables",
    caption=(
"Eight rows of one synthetic dataset and one real credit dataset, shown as heatmaps "
"with the target as the final column separated by a vertical rule. Each feature is "
"rank-normalised within its own column, so shade encodes relative value rather than "
"units."
    ),
);


### C3 · Feature dependence structure

O'Prior's central measurement: the eigenvalue spectrum of the feature correlation matrix — the check
that our changes are not *only* about the target.

**How to read it.** Two priors whose curves coincide teach a similar feature-dependence structure, however different their targets look.

In [ ]:
FIGS.save(
    pp.plot_spectrum_by_variant(loaded),
    "spectrum_by_variant",
    caption=(
"Eigenvalue spectra of the feature correlation matrix for up to 40 synthetic datasets "
"per prior variant, normalised by the largest eigenvalue and plotted against "
"normalised eigenvalue rank. Faint lines are individual datasets; bold lines are the "
"per-variant median."
    ),
);


### C4 · Shape sanity check

Rows and features per synthetic dataset, against the real datasets. Cheap, and it catches a
misconfigured prior at once.

**How to read it.** These should MATCH across priors. A difference here is a confound, not a finding.

In [ ]:
FIGS.save(
    pp.plot_shapes_by_variant(loaded),
    "shapes_by_variant",
    caption=(
"Left: distribution of rows per synthetic dataset. Right: distribution of features "
"per synthetic dataset. One step histogram per prior variant, 20 bins."
    ),
);


## Summary

The priors in text: the two priors and their boundary mass against the real books (A1–A2), then the
realism ranking (A3). Printed last so `output/All_Results.md` carries the numbers, then the
`tfm-library` sources (pin `e5ce016`) and the figure inventory.

In [ ]:
print(summaries.prior_summary(loaded, TASK, source=SOURCE, reference=REFERENCE))
print()
print(summaries.realism_summary(loaded, REAL, TASK))
print()
print(literature.references_md(["outlier_clamp", "quantiles", "kumaraswamy", "oprior_scope", "crps", "filter_rate_reg", "filter_pval", "purucker_missing"]))
print()
print(FIGS.summary())